In [5]:
import math, random
random.seed(42)
block_size=12 
d_model=24 
n_head=4 
head_dim=d_model//n_head
d_ff=4*d_model
lr=0.03 
steps=400 
ln_eps=1e-5


dataset creation 

In [6]:
vocab_size = 20 
# create token names 
itos={i:f"T{i}" for i in range(vocab_size)}
stoi={v:k for k,v in itos.items()}
transitions={}
for i in range(vocab_size):
       next_tokens=random.sample(range(vocab_size),3)
       transitions[i]=next_tokens
data=[]
current=random.randint(0,vocab_size-1)
for _ in range(5000):
       data.append(current)
       if random.random() < 0.8:
              current=random.choice(transitions[current])
       else:
              current=random.randint(0,vocab_size-1)


In [7]:
print(len(data))

5000


In [8]:
print(data[:20])

[14, 17, 1, 19, 8, 0, 8, 18, 2, 17, 1, 1, 7, 12, 12, 18, 10, 4, 6, 6]


In [9]:
def zeros(r, c):
    return [[0.0 for _ in range(c)] for _ in range(r)]

def randn(r, c, scale=0.02):
    return [[(random.random()*2-1)*scale for _ in range(c)] for _ in range(r)]

def vec_zeros(n):
    return [0.0]*n

def matT(A):
    return list(map(list, zip(*A)))

def matmul(A, B):
    r, m = len(A), len(A[0])
    m2, c = len(B), len(B[0])
    assert m == m2
    out = zeros(r, c)
    for i in range(r):
        for k in range(m):
            for j in range(c):
                out[i][j] += A[i][k] * B[k][j]
    return out

def add(A, B):
    r, c = len(A), len(A[0])
    out = zeros(r, c)
    for i in range(r):
        for j in range(c):
            out[i][j] = A[i][j] + B[i][j]
    return out

def scale_mat(A, s):
    r, c = len(A), len(A[0])
    out = zeros(r, c)
    for i in range(r):
        for j in range(c):
            out[i][j] = A[i][j] * s
    return out

def rowwise_softmax(M):
    out = zeros(len(M), len(M[0]))
    for i, row in enumerate(M):
        mx = max(row)
        exps = [math.exp(x-mx) for x in row]
        s = sum(exps)
        out[i] = [e/s for e in exps]
    return out

In [10]:
def gelu(x):
       return 0.5*x*(1+math.tanh(math.sqrt(2/math.pi)*(x+0.044715*x**3)))
def layernorm_forward(x,gamma,beta):
       T=len(x)
       d=len(x[0])
       Y=zeros(t,d)
       cache=[]
       for t in range(T):
              x=x[t]
              mu=sum(x)/d 
              var=sum((v-mu)*(v-mu) for v in x)/d 
              inv=1.0/math.sqrt(var+ln_eps)
              xhat=[(v-mu)*inv for v in x]
              y[t]=[xhat[j]*gamma[j]+beta[j] for j in range(d)]
              Y[t]=y
              cache.append((x,mu,var,xhat,inv))
       return Y,cache


In [11]:
tok_emb=randn(vocab_size,d_model)
pos_emb=randn(block_size,d_model)
Wq=randn(d_model,d_model)
wk=randn(d_model,d_model)
wv=randn(d_model,d_model)   
wo=randn(d_model,d_model)

w1=randn(d_model,d_ff)
w2=randn(d_ff,d_model)


ln1_gamma=[1.0]*d_model
ln1_beta=[0.0]*d_model
ln2_gamma=[1.0]*d_model
ln2_beta=[0.0]*d_model

wlm=randn(d_model,vocab_size)